# K-Means Clustering Implementation

This notebook implements the K-means clustering algorithm with custom functions for updating cluster centers, membership matrices, calculating cost, and visualization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Helper Functions

In [ ]:
def update_cluster_centers(data, membership_matrix):
    """
    Update cluster centers based on current membership matrix.
    
    Parameters:
    -----------
    data : numpy.ndarray
        Data points (n_samples, n_features)
    membership_matrix : numpy.ndarray
        Binary membership matrix (n_samples, n_clusters)
        where membership_matrix[i, j] = 1 if point i belongs to cluster j
    
    Returns:
    --------
    cluster_centers : numpy.ndarray
        Updated cluster centers (n_clusters, n_features)
    """
    num_clusters = membership_matrix.shape[1]
    num_features = data.shape[1]
    cluster_centers = np.zeros((num_clusters, num_features))
    
    for j in range(num_clusters):
        # Get all points assigned to cluster j
        cluster_points = data[membership_matrix[:, j] == 1]
        if len(cluster_points) > 0:
            # Calculate mean of all points in cluster
            cluster_centers[j] = np.mean(cluster_points, axis=0)
        else:
            # If no points assigned to cluster, keep previous center
            cluster_centers[j] = cluster_centers[j]
    
    return cluster_centers

In [ ]:
def update_membership_matrix(data, cluster_centers):
    """
    Update membership matrix by assigning each point to nearest cluster center.
    
    Parameters:
    -----------
    data : numpy.ndarray
        Data points (n_samples, n_features)
    cluster_centers : numpy.ndarray
        Current cluster centers (n_clusters, n_features)
    
    Returns:
    --------
    membership_matrix : numpy.ndarray
        Binary membership matrix (n_samples, n_clusters)
    """
    num_samples = data.shape[0]
    num_clusters = cluster_centers.shape[0]
    membership_matrix = np.zeros((num_samples, num_clusters))
    
    for i in range(num_samples):
        # Calculate distances from point i to all cluster centers
        distances = np.linalg.norm(data[i] - cluster_centers, axis=1)
        # Assign point to nearest cluster
        nearest_cluster = np.argmin(distances)
        membership_matrix[i, nearest_cluster] = 1
    
    return membership_matrix

In [ ]:
def calculate_cost(data, membership_matrix, cluster_centers):
    """
    Calculate the cost (objective function) of current clustering.
    Cost is sum of squared distances from each point to its assigned cluster center.
    
    Parameters:
    -----------
    data : numpy.ndarray
        Data points (n_samples, n_features)
    membership_matrix : numpy.ndarray
        Binary membership matrix (n_samples, n_clusters)
    cluster_centers : numpy.ndarray
        Current cluster centers (n_clusters, n_features)
    
    Returns:
    --------
    cost : float
        Total cost (sum of squared distances)
    """
    cost = 0.0
    num_samples = data.shape[0]
    num_clusters = cluster_centers.shape[0]
    
    for i in range(num_samples):
        for j in range(num_clusters):
            if membership_matrix[i, j] == 1:
                # Add squared distance to cost
                cost += np.sum((data[i] - cluster_centers[j]) ** 2)
    
    return cost

In [ ]:
def plot_k_means(data, membership_matrix, cluster_centers, iteration):
    """
    Visualize the current state of K-means clustering.
    
    Parameters:
    -----------
    data : numpy.ndarray
        Data points (n_samples, n_features)
    membership_matrix : numpy.ndarray
        Binary membership matrix (n_samples, n_clusters)
    cluster_centers : numpy.ndarray
        Current cluster centers (n_clusters, n_features)
    iteration : int
        Current iteration number
    """
    plt.figure(figsize=(8, 6))
    num_clusters = cluster_centers.shape[0]
    colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray']
    
    # Plot data points colored by cluster assignment
    for j in range(num_clusters):
        cluster_points = data[membership_matrix[:, j] == 1]
        if data.shape[1] == 1:
            # 1D data - plot as vertical line
            plt.scatter(cluster_points, np.zeros_like(cluster_points), 
                       c=colors[j % len(colors)], s=100, alpha=0.6, 
                       label=f'Cluster {j+1}')
        elif data.shape[1] == 2:
            # 2D data - regular scatter plot
            plt.scatter(cluster_points[:, 0], cluster_points[:, 1], 
                       c=colors[j % len(colors)], s=100, alpha=0.6, 
                       label=f'Cluster {j+1}')
    
    # Plot cluster centers
    if data.shape[1] == 1:
        plt.scatter(cluster_centers, np.zeros_like(cluster_centers), 
                   c='black', s=300, marker='X', edgecolors='yellow', 
                   linewidths=2, label='Centers', zorder=5)
        plt.ylabel('(Fixed axis for 1D visualization)')
        plt.xlabel('Data values')
    elif data.shape[1] == 2:
        plt.scatter(cluster_centers[:, 0], cluster_centers[:, 1], 
                   c='black', s=300, marker='X', edgecolors='yellow', 
                   linewidths=2, label='Centers', zorder=5)
        plt.xlabel('Feature 1')
        plt.ylabel('Feature 2')
    
    plt.title(f'K-Means Clustering - Iteration {iteration}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Example Usage

Below is an example of how to use the K-means implementation with custom initial centroids.

In [ ]:
# Generate Data
new_data = np.array([[1], [2], [5], [8]], dtype=float)

# Parameters
num_clusters = 2 # Number of clusters

# Initial Cost
cost_prev = 0

# Convergence
epsilon = 1e-5
max_iter = 2

# Run K C-Means
#Option 1: With initial centroids determined by randomly initialized centroids
#membership_matrix, cluster_centers = k_means(new_data, num_clusters, max_iter=2)

#Option 2: With self-defined intial centroids
initial_cluster_centers = np.array([[2], [7]]) #Initial Centroids

for iteration in range(max_iter):
        # Update cluster centers
        if iteration == 0:
          cluster_centers = initial_cluster_centers
        else:
          cluster_centers = update_cluster_centers(new_data, membership_matrix)
        
        # Update membership values
        membership_matrix = update_membership_matrix(new_data, cluster_centers)
        
        # Calculate the cost (objective function)
        cost = calculate_cost(new_data, membership_matrix, cluster_centers)
        
        # Plot the current state
        plot_k_means(new_data, membership_matrix, cluster_centers, iteration)
        
        print("Cluster centers:", cluster_centers.flatten())
        print("Memberships:\n", membership_matrix)
        print("Cost:\n", cost)
        
        # Check for convergence
        if abs(cost - cost_prev) <= epsilon:
            print(f"Converged at iteration {iteration}")
            break
        cost_prev = cost